# nuScenes Minimal Visualization
Front camera + LiDAR + 3D Bounding Box visualization **without nuscenes-devkit**.

- Scenes with **bus, bicycle/motorcycle/pedestrian, car**
- 3D bbox projected onto camera image
- 3D point cloud with bounding boxes (BEV + 3D scatter)

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from collections import defaultdict
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

DATAROOT = '/data/nuscenes'
VERSION = 'v1.0-trainval'

## 1. Load nuScenes metadata (JSON only)

In [ ]:
def load_table(name):
    with open(os.path.join(DATAROOT, VERSION, f'{name}.json')) as f:
        return json.load(f)

# Load all needed tables
scenes = load_table('scene')
samples = load_table('sample')
sample_datas = load_table('sample_data')
sample_anns = load_table('sample_annotation')
instances = load_table('instance')
categories = load_table('category')
calibrated_sensors = load_table('calibrated_sensor')
ego_poses = load_table('ego_pose')
sensors = load_table('sensor')

# Build token -> record lookups
def index_by_token(records):
    return {r['token']: r for r in records}

scene_idx = index_by_token(scenes)
sample_idx = index_by_token(samples)
sd_idx = index_by_token(sample_datas)
cs_idx = index_by_token(calibrated_sensors)
ep_idx = index_by_token(ego_poses)
inst_idx = index_by_token(instances)
cat_idx = index_by_token(categories)
sensor_idx = index_by_token(sensors)

# Category name lookup
def get_category(ann):
    inst = inst_idx[ann['instance_token']]
    return cat_idx[inst['category_token']]['name']

print(f'Loaded: {len(scenes)} scenes, {len(samples)} samples, {len(sample_anns)} annotations')

## 2. Find target scenes (bus + person/bike + car)

In [ ]:
# Group sample_data by (sample_token, channel)
sd_by_sample_channel = {}
for sd in sample_datas:
    if not sd['is_key_frame']:
        continue
    cs = cs_idx[sd['calibrated_sensor_token']]
    ch = sensor_idx[cs['sensor_token']]['channel']
    sd_by_sample_channel[(sd['sample_token'], ch)] = sd

# Group annotations by sample
anns_by_sample = defaultdict(list)
for ann in sample_anns:
    anns_by_sample[ann['sample_token']].append(ann)

# Target categories
BUS = {'vehicle.bus.bendy', 'vehicle.bus.rigid'}
PERSON_BIKE = {'vehicle.bicycle', 'vehicle.motorcycle',
               'human.pedestrian.adult', 'human.pedestrian.child',
               'human.pedestrian.construction_worker', 'human.pedestrian.personal_mobility'}
CAR = {'vehicle.car'}

# Find samples with all three categories at once
target_samples = []
for stok, s_anns in anns_by_sample.items():
    cats = {get_category(a) for a in s_anns}
    if (cats & BUS) and (cats & PERSON_BIKE) and (cats & CAR):
        target_samples.append(stok)

print(f'Found {len(target_samples)} samples with bus + person/bike + car')

# Pick 2 good scenes - scene-0523 and scene-0916
scene_name_map = {s['name']: s for s in scenes}
chosen_scenes = ['scene-0523', 'scene-0916']

# Fallback: if these scenes don't have target samples, pick from target_samples
chosen_tokens = []
for sname in chosen_scenes:
    if sname not in scene_name_map:
        continue
    sc = scene_name_map[sname]
    # Find a target sample in this scene
    scene_target = [t for t in target_samples if sample_idx[t]['scene_token'] == sc['token']]
    if scene_target:
        chosen_tokens.append(scene_target[len(scene_target)//2])  # pick middle frame

# If not enough, pick from target_samples directly
if len(chosen_tokens) < 2:
    for t in target_samples:
        if t not in chosen_tokens:
            chosen_tokens.append(t)
        if len(chosen_tokens) >= 2:
            break

for tok in chosen_tokens:
    s = sample_idx[tok]
    sc = scene_idx[s['scene_token']]
    cats = {get_category(a) for a in anns_by_sample[tok]}
    print(f"\n{sc['name']}: {sc['description']}")
    print(f"  Categories: {sorted(cats)}")

## 3. Coordinate transform utilities

In [ ]:
def quaternion_to_rotation_matrix(q):
    """Convert quaternion [w, x, y, z] to 3x3 rotation matrix."""
    w, x, y, z = q
    return np.array([
        [1 - 2*(y*y + z*z), 2*(x*y - w*z),     2*(x*z + w*y)],
        [2*(x*y + w*z),     1 - 2*(x*x + z*z), 2*(y*z - w*x)],
        [2*(x*z - w*y),     2*(y*z + w*x),     1 - 2*(x*x + y*y)]
    ])

def make_transform(translation, rotation):
    """Build 4x4 homogeneous transform from translation + quaternion."""
    T = np.eye(4)
    T[:3, :3] = quaternion_to_rotation_matrix(rotation)
    T[:3, 3] = translation
    return T

def get_sensor_transform(sd_record):
    """Get global->sensor transform for a sample_data record."""
    cs = cs_idx[sd_record['calibrated_sensor_token']]
    ep = ep_idx[sd_record['ego_pose_token']]
    # ego -> global
    ego2global = make_transform(ep['translation'], ep['rotation'])
    # sensor -> ego
    sensor2ego = make_transform(cs['translation'], cs['rotation'])
    # sensor -> global = ego2global @ sensor2ego
    sensor2global = ego2global @ sensor2ego
    # global -> sensor
    global2sensor = np.linalg.inv(sensor2global)
    return global2sensor, cs

def get_3d_box_corners(ann):
    """Get 8 corners of 3D bounding box in global frame.
    Size: [width, length, height]. Center at annotation translation.
    """
    w, l, h = ann['size']  # width, length, height
    # Corners in object frame (center at origin, z-axis up)
    # Bottom 4 corners, then top 4 corners
    x = l / 2
    y = w / 2
    corners = np.array([
        [ x,  y, 0], [ x, -y, 0], [-x, -y, 0], [-x,  y, 0],  # bottom
        [ x,  y, h], [ x, -y, h], [-x, -y, h], [-x,  y, h],  # top
    ])  # (8, 3)
    # Shift z so center is at center height
    corners[:, 2] -= h / 2
    
    # Rotate to global
    R = quaternion_to_rotation_matrix(ann['rotation'])
    corners = (R @ corners.T).T  # (8, 3)
    # Translate to global
    corners += np.array(ann['translation'])
    return corners

def project_to_image(points_3d, intrinsic):
    """Project 3D points (N,3) to image using 3x3 camera intrinsic. Returns (N,2) pixel coords."""
    pts = intrinsic @ points_3d.T  # (3, N)
    pts[:2] /= pts[2:3]  # normalize
    return pts[:2].T, pts[2]  # (N,2) pixel coords, (N,) depths

print('Transform utilities loaded.')

## 4. Data loading utilities

In [ ]:
def load_image(sd_record):
    """Load image from sample_data record."""
    path = os.path.join(DATAROOT, sd_record['filename'])
    return np.array(Image.open(path))

def load_lidar(sd_record):
    """Load LiDAR pointcloud from .bin file. Returns (N, 5): x, y, z, intensity, ring."""
    path = os.path.join(DATAROOT, sd_record['filename'])
    points = np.fromfile(path, dtype=np.float32).reshape(-1, 5)
    return points

# Color scheme for categories
CAT_COLORS = {
    'vehicle.bus.bendy': '#e6194b',
    'vehicle.bus.rigid': '#e6194b',
    'vehicle.car': '#3cb44b',
    'vehicle.truck': '#4363d8',
    'vehicle.trailer': '#911eb4',
    'vehicle.construction': '#f58231',
    'vehicle.bicycle': '#42d4f4',
    'vehicle.motorcycle': '#f032e6',
    'vehicle.emergency.police': '#bfef45',
    'vehicle.emergency.ambulance': '#bfef45',
    'human.pedestrian.adult': '#fabed4',
    'human.pedestrian.child': '#fabed4',
    'human.pedestrian.construction_worker': '#ffd8b1',
    'human.pedestrian.personal_mobility': '#dcbeff',
    'human.pedestrian.police_officer': '#aaffc3',
}

# Simplified display names
CAT_DISPLAY = {
    'vehicle.bus.bendy': 'Bus', 'vehicle.bus.rigid': 'Bus',
    'vehicle.car': 'Car', 'vehicle.truck': 'Truck',
    'vehicle.trailer': 'Trailer', 'vehicle.construction': 'Construction',
    'vehicle.bicycle': 'Bicycle', 'vehicle.motorcycle': 'Motorcycle',
    'vehicle.emergency.police': 'Police', 'vehicle.emergency.ambulance': 'Ambulance',
    'human.pedestrian.adult': 'Pedestrian', 'human.pedestrian.child': 'Child',
    'human.pedestrian.construction_worker': 'Worker',
    'human.pedestrian.personal_mobility': 'Scooter',
    'human.pedestrian.police_officer': 'Police Officer',
}

def get_color(cat):
    return CAT_COLORS.get(cat, '#808080')

def get_display_name(cat):
    return CAT_DISPLAY.get(cat, cat.split('.')[-1])

print('Data loading utilities ready.')

## 5. Visualization: 3D BBox projected on front camera image

In [ ]:
# Edges connecting box corners (bottom face, top face, vertical pillars)
BOX_EDGES = [
    (0,1),(1,2),(2,3),(3,0),  # bottom
    (4,5),(5,6),(6,7),(7,4),  # top
    (0,4),(1,5),(2,6),(3,7),  # pillars
]

def clip_line_to_rect(x0, y0, x1, y1, xmin, ymin, xmax, ymax):
    """Cohen-Sutherland line clipping. Returns clipped (x0,y0,x1,y1) or None."""
    INSIDE, LEFT, RIGHT, BOTTOM, TOP = 0, 1, 2, 4, 8
    def code(x, y):
        c = INSIDE
        if x < xmin: c |= LEFT
        elif x > xmax: c |= RIGHT
        if y < ymin: c |= BOTTOM
        elif y > ymax: c |= TOP
        return c
    c0, c1 = code(x0, y0), code(x1, y1)
    for _ in range(20):
        if not (c0 | c1):
            return x0, y0, x1, y1
        if c0 & c1:
            return None
        c = c0 or c1
        if c & TOP:
            x = x0 + (x1-x0)*(ymax-y0)/(y1-y0); y = ymax
        elif c & BOTTOM:
            x = x0 + (x1-x0)*(ymin-y0)/(y1-y0); y = ymin
        elif c & RIGHT:
            y = y0 + (y1-y0)*(xmax-x0)/(x1-x0); x = xmax
        elif c & LEFT:
            y = y0 + (y1-y0)*(xmin-x0)/(x1-x0); x = xmin
        if c == c0:
            x0, y0, c0 = x, y, code(x, y)
        else:
            x1, y1, c1 = x, y, code(x, y)
    return None

def draw_box_on_image(ax, corners_2d, color, linewidth=1.5, img_w=1600, img_h=900):
    """Draw projected 3D box edges on image, clipped to image bounds."""
    margin = 50
    for i, j in BOX_EDGES:
        result = clip_line_to_rect(
            corners_2d[i,0], corners_2d[i,1], corners_2d[j,0], corners_2d[j,1],
            -margin, -margin, img_w+margin, img_h+margin)
        if result is not None:
            ax.plot([result[0], result[2]], [result[1], result[3]],
                    color=color, linewidth=linewidth)

def visualize_camera_with_boxes(sample_token, save_path=None):
    """Visualize front camera image with projected 3D bounding boxes."""
    cam_sd = sd_by_sample_channel.get((sample_token, 'CAM_FRONT'))
    if cam_sd is None:
        print('No CAM_FRONT data found'); return
    
    img = load_image(cam_sd)
    h, w = img.shape[:2]
    
    global2cam, cam_cs = get_sensor_transform(cam_sd)
    intrinsic = np.array(cam_cs['camera_intrinsic'])
    
    fig, ax = plt.subplots(1, 1, figsize=(16, 9))
    ax.imshow(img)
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)
    
    legend_entries = {}
    anns = anns_by_sample[sample_token]
    
    for ann in anns:
        cat = get_category(ann)
        color = get_color(cat)
        corners_global = get_3d_box_corners(ann)
        corners_hom = np.hstack([corners_global, np.ones((8,1))])
        corners_cam = (global2cam @ corners_hom.T).T[:, :3]
        
        # Skip if any corner behind camera
        if np.any(corners_cam[:, 2] <= 0.5):
            continue
        
        corners_2d, depths = project_to_image(corners_cam, intrinsic)
        
        # Skip if entirely outside image (with margin)
        margin = 200
        if (corners_2d[:, 0].max() < -margin or corners_2d[:, 0].min() > w + margin or
            corners_2d[:, 1].max() < -margin or corners_2d[:, 1].min() > h + margin):
            continue
        
        draw_box_on_image(ax, corners_2d, color, img_w=w, img_h=h)
        display = get_display_name(cat)
        if display not in legend_entries:
            legend_entries[display] = color
        
        # Label
        dist = np.linalg.norm(corners_cam.mean(axis=0))
        cx, cy = np.clip(corners_2d.mean(axis=0), [0, 0], [w, h])
        if 10 <= cx <= w-10 and 10 <= cy <= h-10:
            ax.text(cx, cy - 8, f'{display} {dist:.0f}m',
                    color='white', fontsize=7, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.7))
    
    patches = [mpatches.Patch(color=c, label=n) for n, c in legend_entries.items()]
    ax.legend(handles=patches, loc='upper right', fontsize=8)
    
    sc = scene_idx[sample_idx[sample_token]['scene_token']]
    ax.set_title(f"{sc['name']} - Front Camera + 3D BBox\n{sc['description']}", fontsize=12)
    ax.axis('off')
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches='tight', dpi=150)
        print(f'Saved: {save_path}')
    plt.show()

print('Camera visualization function ready.')

## 6. Visualization: LiDAR point cloud with 3D bounding boxes

In [ ]:
def visualize_lidar_with_boxes(sample_token, save_path=None, view='bev', xlim=(-50,50), ylim=(-50,50)):
    """Visualize LiDAR point cloud with 3D bounding boxes.
    view: 'bev' for top-down, '3d' for 3D scatter.
    """
    lidar_sd = sd_by_sample_channel.get((sample_token, 'LIDAR_TOP'))
    if lidar_sd is None:
        print('No LIDAR_TOP data found'); return
    
    # Load point cloud
    pc = load_lidar(lidar_sd)  # (N, 5)
    
    # LiDAR transform
    global2lidar, _ = get_sensor_transform(lidar_sd)
    
    # Annotations
    anns = anns_by_sample[sample_token]
    
    if view == 'bev':
        fig, ax = plt.subplots(1, 1, figsize=(12, 12))
        # Plot points (in lidar frame = already lidar coordinates for .bin)
        mask = (pc[:, 0] > xlim[0]) & (pc[:, 0] < xlim[1]) & (pc[:, 1] > ylim[0]) & (pc[:, 1] < ylim[1])
        ax.scatter(pc[mask, 0], pc[mask, 1], c=pc[mask, 2], cmap='viridis',
                   s=0.3, alpha=0.5, vmin=-3, vmax=2)
        
        legend_entries = {}
        for ann in anns:
            cat = get_category(ann)
            color = get_color(cat)
            corners_global = get_3d_box_corners(ann)  # (8, 3)
            
            # Transform to lidar frame
            corners_hom = np.hstack([corners_global, np.ones((8,1))])
            corners_lidar = (global2lidar @ corners_hom.T).T[:, :3]
            
            # Draw bottom face in BEV
            bottom = corners_lidar[:4]  # bottom 4 corners
            polygon = plt.Polygon(bottom[:, :2], fill=False, edgecolor=color, linewidth=1.5)
            ax.add_patch(polygon)
            
            display = get_display_name(cat)
            if display not in legend_entries:
                legend_entries[display] = color
            
            # Label
            cx, cy = bottom[:, :2].mean(axis=0)
            if xlim[0] < cx < xlim[1] and ylim[0] < cy < ylim[1]:
                ax.text(cx, cy, display, color='white', fontsize=6,
                        ha='center', va='center',
                        bbox=dict(boxstyle='round,pad=0.15', facecolor=color, alpha=0.7))
        
        # Ego vehicle
        ego_rect = plt.Rectangle((-1, -2), 2, 4, fill=True, facecolor='yellow', edgecolor='yellow', alpha=0.8)
        ax.add_patch(ego_rect)
        ax.text(0, 0, 'EGO', ha='center', va='center', fontsize=7, fontweight='bold')
        
        patches = [mpatches.Patch(color=c, label=n) for n, c in legend_entries.items()]
        ax.legend(handles=patches, loc='upper right', fontsize=8)
        
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_aspect('equal')
        ax.set_facecolor('#1a1a2e')
        fig.patch.set_facecolor('#1a1a2e')
        ax.tick_params(colors='white')
        for spine in ax.spines.values():
            spine.set_color('white')
        
        sc = scene_idx[sample_idx[sample_token]['scene_token']]
        ax.set_title(f"{sc['name']} - LiDAR BEV + 3D BBox\n{sc['description']}",
                     fontsize=12, color='white')
        ax.set_xlabel('X (m)', color='white')
        ax.set_ylabel('Y (m)', color='white')
    
    elif view == '3d':
        fig = plt.figure(figsize=(14, 10))
        ax = fig.add_subplot(111, projection='3d')
        
        # Subsample for speed
        mask = (pc[:, 0] > xlim[0]) & (pc[:, 0] < xlim[1]) & (pc[:, 1] > ylim[0]) & (pc[:, 1] < ylim[1])
        pts = pc[mask]
        step = max(1, len(pts) // 30000)
        pts = pts[::step]
        
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=pts[:, 2], cmap='viridis',
                   s=0.2, alpha=0.3, vmin=-3, vmax=2)
        
        legend_entries = {}
        for ann in anns:
            cat = get_category(ann)
            color = get_color(cat)
            corners_global = get_3d_box_corners(ann)
            corners_hom = np.hstack([corners_global, np.ones((8,1))])
            corners_lidar = (global2lidar @ corners_hom.T).T[:, :3]
            
            # Draw all 12 edges
            for i, j in BOX_EDGES:
                ax.plot3D([corners_lidar[i,0], corners_lidar[j,0]],
                          [corners_lidar[i,1], corners_lidar[j,1]],
                          [corners_lidar[i,2], corners_lidar[j,2]],
                          color=color, linewidth=1.0)
            
            display = get_display_name(cat)
            if display not in legend_entries:
                legend_entries[display] = color
        
        patches = [mpatches.Patch(color=c, label=n) for n, c in legend_entries.items()]
        ax.legend(handles=patches, loc='upper right', fontsize=8)
        
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_zlim(-3, 5)
        ax.set_xlabel('X (m)')
        ax.set_ylabel('Y (m)')
        ax.set_zlabel('Z (m)')
        ax.view_init(elev=30, azim=-60)
        
        sc = scene_idx[sample_idx[sample_token]['scene_token']]
        ax.set_title(f"{sc['name']} - LiDAR 3D + BBox\n{sc['description']}", fontsize=12)
    
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches='tight', dpi=150, facecolor=fig.get_facecolor())
        print(f'Saved: {save_path}')
    plt.show()

print('LiDAR visualization function ready.')

## 7. Visualization: LiDAR points projected on front camera

In [ ]:
def visualize_lidar_on_image(sample_token, save_path=None):
    """Project LiDAR points onto front camera image, colored by depth."""
    cam_sd = sd_by_sample_channel.get((sample_token, 'CAM_FRONT'))
    lidar_sd = sd_by_sample_channel.get((sample_token, 'LIDAR_TOP'))
    if cam_sd is None or lidar_sd is None:
        print('Missing data'); return
    
    img = load_image(cam_sd)
    h, w = img.shape[:2]
    pc = load_lidar(lidar_sd)  # (N, 5) in lidar frame
    
    # LiDAR -> global -> camera
    lidar_cs = cs_idx[lidar_sd['calibrated_sensor_token']]
    lidar_ep = ep_idx[lidar_sd['ego_pose_token']]
    lidar2ego = make_transform(lidar_cs['translation'], lidar_cs['rotation'])
    ego2global_lidar = make_transform(lidar_ep['translation'], lidar_ep['rotation'])
    lidar2global = ego2global_lidar @ lidar2ego
    
    cam_cs = cs_idx[cam_sd['calibrated_sensor_token']]
    cam_ep = ep_idx[cam_sd['ego_pose_token']]
    cam2ego = make_transform(cam_cs['translation'], cam_cs['rotation'])
    ego2global_cam = make_transform(cam_ep['translation'], cam_ep['rotation'])
    cam2global = ego2global_cam @ cam2ego
    global2cam = np.linalg.inv(cam2global)
    
    # Full transform: lidar -> camera
    lidar2cam = global2cam @ lidar2global
    intrinsic = np.array(cam_cs['camera_intrinsic'])
    
    # Transform points
    pts_hom = np.hstack([pc[:, :3], np.ones((len(pc), 1))])  # (N, 4)
    pts_cam = (lidar2cam @ pts_hom.T).T[:, :3]  # (N, 3)
    
    # Keep only points in front of camera
    mask = pts_cam[:, 2] > 1.0
    pts_cam = pts_cam[mask]
    
    # Project
    pts_2d, depths = project_to_image(pts_cam, intrinsic)
    
    # Filter to image bounds
    mask = (pts_2d[:, 0] >= 0) & (pts_2d[:, 0] < w) & (pts_2d[:, 1] >= 0) & (pts_2d[:, 1] < h)
    pts_2d = pts_2d[mask]
    depths = depths[mask]
    
    fig, ax = plt.subplots(1, 1, figsize=(16, 9))
    ax.imshow(img)
    sc = ax.scatter(pts_2d[:, 0], pts_2d[:, 1], c=depths, cmap='turbo',
                    s=1.0, alpha=0.7, vmin=1, vmax=60)
    plt.colorbar(sc, ax=ax, label='Depth (m)', shrink=0.6)
    
    scene = scene_idx[sample_idx[sample_token]['scene_token']]
    ax.set_title(f"{scene['name']} - LiDAR on Camera\n{scene['description']}", fontsize=12)
    ax.axis('off')
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches='tight', dpi=150)
        print(f'Saved: {save_path}')
    plt.show()

print('LiDAR-on-image projection function ready.')

## 8. Run all visualizations for chosen samples

In [ ]:
SAVE_DIR = '/home/dohyun/e2e/nuscenes_vis/output'
os.makedirs(SAVE_DIR, exist_ok=True)

for i, tok in enumerate(chosen_tokens):
    sc = scene_idx[sample_idx[tok]['scene_token']]
    prefix = sc['name'].replace('-', '_')
    print(f"\n{'='*60}")
    print(f"Visualizing {sc['name']}: {sc['description']}")
    print(f"{'='*60}")
    
    # 1) Front camera + 3D bbox
    visualize_camera_with_boxes(tok, save_path=os.path.join(SAVE_DIR, f'{prefix}_cam_bbox.png'))
    
    # 2) LiDAR BEV + bbox
    visualize_lidar_with_boxes(tok, save_path=os.path.join(SAVE_DIR, f'{prefix}_lidar_bev.png'), view='bev')
    
    # 3) LiDAR 3D + bbox
    visualize_lidar_with_boxes(tok, save_path=os.path.join(SAVE_DIR, f'{prefix}_lidar_3d.png'), view='3d')
    
    # 4) LiDAR projected on camera
    visualize_lidar_on_image(tok, save_path=os.path.join(SAVE_DIR, f'{prefix}_lidar_on_cam.png'))

print(f'\nAll outputs saved to: {SAVE_DIR}')

## 9. Multi-frame sequence visualization (video-style)

In [ ]:
def get_scene_samples(scene_token):
    """Get all sample tokens in a scene, ordered by time."""
    sc = scene_idx[scene_token]
    result = []
    tok = sc['first_sample_token']
    while tok:
        result.append(tok)
        tok = sample_idx[tok]['next']
        if tok == '':
            break
    return result

# Show 8 evenly-spaced frames from the first chosen scene
first_scene_tok = sample_idx[chosen_tokens[0]]['scene_token']
all_samples = get_scene_samples(first_scene_tok)
n_frames = min(8, len(all_samples))
indices = np.linspace(0, len(all_samples)-1, n_frames, dtype=int)

fig, axes = plt.subplots(2, 4, figsize=(24, 10))
sc = scene_idx[first_scene_tok]

for idx, ax in zip(indices, axes.flat):
    stok = all_samples[idx]
    cam_sd = sd_by_sample_channel.get((stok, 'CAM_FRONT'))
    if cam_sd is None:
        continue
    img = load_image(cam_sd)
    global2cam, cam_cs = get_sensor_transform(cam_sd)
    intrinsic = np.array(cam_cs['camera_intrinsic'])
    h, w = img.shape[:2]
    
    ax.imshow(img)
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)
    for ann in anns_by_sample[stok]:
        cat = get_category(ann)
        color = get_color(cat)
        corners_global = get_3d_box_corners(ann)
        corners_hom = np.hstack([corners_global, np.ones((8,1))])
        corners_cam = (global2cam @ corners_hom.T).T[:, :3]
        if np.any(corners_cam[:, 2] <= 0.5):
            continue
        corners_2d, _ = project_to_image(corners_cam, intrinsic)
        if (corners_2d[:, 0].max() < -200 or corners_2d[:, 0].min() > w + 200 or
            corners_2d[:, 1].max() < -200 or corners_2d[:, 1].min() > h + 200):
            continue
        draw_box_on_image(ax, corners_2d, color, linewidth=1.0, img_w=w, img_h=h)
    
    ax.set_title(f'Frame {idx}/{len(all_samples)-1}', fontsize=10)
    ax.axis('off')

fig.suptitle(f"{sc['name']} - Sequence ({n_frames} frames)\n{sc['description']}", fontsize=14)
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, f'{sc["name"].replace("-","_")}_sequence.png'),
            bbox_inches='tight', dpi=150)
plt.show()
print('Sequence visualization saved.')

## 10. Video generation: Camera + 3D BBox + BEV (side-by-side)

In [ ]:
import cv2
from matplotlib.colors import to_rgb
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for rendering

def fig_to_array(fig):
    """Render matplotlib figure to numpy RGB array."""
    fig.canvas.draw()
    buf = fig.canvas.buffer_rgba()
    arr = np.asarray(buf)
    return arr[:, :, :3].copy()  # drop alpha

def render_cam_bbox_frame(sample_token, fig_w=12, fig_h=6.75):
    """Render a single camera+bbox frame as numpy array."""
    cam_sd = sd_by_sample_channel.get((sample_token, 'CAM_FRONT'))
    if cam_sd is None:
        return None
    
    img = load_image(cam_sd)
    h, w = img.shape[:2]
    global2cam, cam_cs = get_sensor_transform(cam_sd)
    intrinsic = np.array(cam_cs['camera_intrinsic'])
    
    fig, ax = plt.subplots(1, 1, figsize=(fig_w, fig_h), dpi=128)
    ax.imshow(img)
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)
    
    legend_entries = {}
    for ann in anns_by_sample[sample_token]:
        cat = get_category(ann)
        color = get_color(cat)
        corners_global = get_3d_box_corners(ann)
        corners_hom = np.hstack([corners_global, np.ones((8,1))])
        corners_cam = (global2cam @ corners_hom.T).T[:, :3]
        if np.any(corners_cam[:, 2] <= 0.5):
            continue
        corners_2d, depths = project_to_image(corners_cam, intrinsic)
        margin = 200
        if (corners_2d[:, 0].max() < -margin or corners_2d[:, 0].min() > w + margin or
            corners_2d[:, 1].max() < -margin or corners_2d[:, 1].min() > h + margin):
            continue
        draw_box_on_image(ax, corners_2d, color, img_w=w, img_h=h)
        display = get_display_name(cat)
        if display not in legend_entries:
            legend_entries[display] = color
        dist = np.linalg.norm(corners_cam.mean(axis=0))
        cx, cy = np.clip(corners_2d.mean(axis=0), [0, 0], [w, h])
        if 10 <= cx <= w-10 and 10 <= cy <= h-10:
            ax.text(cx, cy - 8, f'{display} {dist:.0f}m',
                    color='white', fontsize=6, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.15', facecolor=color, alpha=0.7))
    
    patches = [mpatches.Patch(color=c, label=n) for n, c in legend_entries.items()]
    if patches:
        ax.legend(handles=patches, loc='upper right', fontsize=7)
    ax.axis('off')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    
    frame = fig_to_array(fig)
    plt.close(fig)
    return frame

def render_bev_frame(sample_token, fig_size=6.75, xlim=(-50,50), ylim=(-50,50)):
    """Render a single BEV frame as numpy array."""
    lidar_sd = sd_by_sample_channel.get((sample_token, 'LIDAR_TOP'))
    if lidar_sd is None:
        return None
    
    pc = load_lidar(lidar_sd)
    global2lidar, _ = get_sensor_transform(lidar_sd)
    
    fig, ax = plt.subplots(1, 1, figsize=(fig_size, fig_size), dpi=128)
    mask = (pc[:, 0] > xlim[0]) & (pc[:, 0] < xlim[1]) & (pc[:, 1] > ylim[0]) & (pc[:, 1] < ylim[1])
    ax.scatter(pc[mask, 0], pc[mask, 1], c=pc[mask, 2], cmap='viridis',
               s=0.2, alpha=0.5, vmin=-3, vmax=2)
    
    for ann in anns_by_sample[sample_token]:
        cat = get_category(ann)
        color = get_color(cat)
        corners_global = get_3d_box_corners(ann)
        corners_hom = np.hstack([corners_global, np.ones((8,1))])
        corners_lidar = (global2lidar @ corners_hom.T).T[:, :3]
        bottom = corners_lidar[:4]
        polygon = plt.Polygon(bottom[:, :2], fill=False, edgecolor=color, linewidth=1.5)
        ax.add_patch(polygon)
    
    ego_rect = plt.Rectangle((-1, -2), 2, 4, fill=True, facecolor='yellow', edgecolor='yellow', alpha=0.8)
    ax.add_patch(ego_rect)
    
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal')
    ax.set_facecolor('#1a1a2e')
    fig.patch.set_facecolor('#1a1a2e')
    ax.tick_params(colors='white', labelsize=7)
    for spine in ax.spines.values():
        spine.set_color('white')
    ax.set_xlabel('X (m)', color='white', fontsize=8)
    ax.set_ylabel('Y (m)', color='white', fontsize=8)
    plt.tight_layout(pad=0.5)
    
    frame = fig_to_array(fig)
    plt.close(fig)
    return frame

def render_lidar_on_cam_frame(sample_token, fig_w=12, fig_h=6.75):
    """Render LiDAR-on-camera projection as numpy array."""
    cam_sd = sd_by_sample_channel.get((sample_token, 'CAM_FRONT'))
    lidar_sd = sd_by_sample_channel.get((sample_token, 'LIDAR_TOP'))
    if cam_sd is None or lidar_sd is None:
        return None
    
    img = load_image(cam_sd)
    h, w = img.shape[:2]
    pc = load_lidar(lidar_sd)
    
    lidar_cs = cs_idx[lidar_sd['calibrated_sensor_token']]
    lidar_ep = ep_idx[lidar_sd['ego_pose_token']]
    lidar2global = make_transform(lidar_ep['translation'], lidar_ep['rotation']) @ \
                   make_transform(lidar_cs['translation'], lidar_cs['rotation'])
    
    cam_cs = cs_idx[cam_sd['calibrated_sensor_token']]
    cam_ep = ep_idx[cam_sd['ego_pose_token']]
    cam2global = make_transform(cam_ep['translation'], cam_ep['rotation']) @ \
                 make_transform(cam_cs['translation'], cam_cs['rotation'])
    lidar2cam = np.linalg.inv(cam2global) @ lidar2global
    intrinsic = np.array(cam_cs['camera_intrinsic'])
    
    pts_hom = np.hstack([pc[:, :3], np.ones((len(pc), 1))])
    pts_cam = (lidar2cam @ pts_hom.T).T[:, :3]
    mask = pts_cam[:, 2] > 1.0
    pts_cam = pts_cam[mask]
    pts_2d, depths = project_to_image(pts_cam, intrinsic)
    mask = (pts_2d[:, 0] >= 0) & (pts_2d[:, 0] < w) & (pts_2d[:, 1] >= 0) & (pts_2d[:, 1] < h)
    pts_2d, depths = pts_2d[mask], depths[mask]
    
    fig, ax = plt.subplots(1, 1, figsize=(fig_w, fig_h), dpi=128)
    ax.imshow(img)
    ax.scatter(pts_2d[:, 0], pts_2d[:, 1], c=depths, cmap='turbo',
               s=0.8, alpha=0.7, vmin=1, vmax=60)
    ax.axis('off')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    
    frame = fig_to_array(fig)
    plt.close(fig)
    return frame

print('Video frame renderers ready.')

In [ ]:
def make_video_sidebyside(scene_token, output_path, fps=2):
    """Generate side-by-side video: Camera+3D BBox (left) | BEV (right)."""
    sample_tokens = get_scene_samples(scene_token)
    sc = scene_idx[scene_token]
    print(f"Generating side-by-side video for {sc['name']} ({len(sample_tokens)} frames)...")
    
    writer = None
    for i, stok in enumerate(sample_tokens):
        cam_frame = render_cam_bbox_frame(stok)
        bev_frame = render_bev_frame(stok)
        if cam_frame is None or bev_frame is None:
            continue
        
        # Resize BEV to match camera height
        cam_h, cam_w = cam_frame.shape[:2]
        bev_h, bev_w = bev_frame.shape[:2]
        scale = cam_h / bev_h
        new_bev_w = int(bev_w * scale)
        bev_resized = cv2.resize(bev_frame, (new_bev_w, cam_h))
        
        # Concatenate side by side
        composite = np.hstack([cam_frame, bev_resized])
        
        if writer is None:
            comp_h, comp_w = composite.shape[:2]
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            writer = cv2.VideoWriter(output_path, fourcc, fps, (comp_w, comp_h))
        
        writer.write(cv2.cvtColor(composite, cv2.COLOR_RGB2BGR))
        if (i + 1) % 10 == 0:
            print(f"  Frame {i+1}/{len(sample_tokens)}")
    
    if writer:
        writer.release()
        print(f"  Saved: {output_path}")
    return output_path

def make_video_cam_bbox(scene_token, output_path, fps=2):
    """Generate camera + 3D bbox video."""
    sample_tokens = get_scene_samples(scene_token)
    sc = scene_idx[scene_token]
    print(f"Generating camera+bbox video for {sc['name']} ({len(sample_tokens)} frames)...")
    
    writer = None
    for i, stok in enumerate(sample_tokens):
        frame = render_cam_bbox_frame(stok)
        if frame is None:
            continue
        if writer is None:
            fh, fw = frame.shape[:2]
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            writer = cv2.VideoWriter(output_path, fourcc, fps, (fw, fh))
        writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    
    if writer:
        writer.release()
        print(f"  Saved: {output_path}")
    return output_path

def make_video_lidar_on_cam(scene_token, output_path, fps=2):
    """Generate LiDAR projected on camera video."""
    sample_tokens = get_scene_samples(scene_token)
    sc = scene_idx[scene_token]
    print(f"Generating LiDAR-on-camera video for {sc['name']} ({len(sample_tokens)} frames)...")
    
    writer = None
    for i, stok in enumerate(sample_tokens):
        frame = render_lidar_on_cam_frame(stok)
        if frame is None:
            continue
        if writer is None:
            fh, fw = frame.shape[:2]
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            writer = cv2.VideoWriter(output_path, fourcc, fps, (fw, fh))
        writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    
    if writer:
        writer.release()
        print(f"  Saved: {output_path}")
    return output_path

print('Video generation functions ready.')

In [ ]:
VIDEO_DIR = os.path.join(SAVE_DIR, 'videos')
os.makedirs(VIDEO_DIR, exist_ok=True)

for tok in chosen_tokens:
    scene_tok = sample_idx[tok]['scene_token']
    sc = scene_idx[scene_tok]
    prefix = sc['name'].replace('-', '_')
    
    print(f"\n{'='*60}")
    print(f"Videos for {sc['name']}: {sc['description']}")
    print(f"{'='*60}")
    
    # 1) Camera + 3D BBox video
    make_video_cam_bbox(scene_tok, os.path.join(VIDEO_DIR, f'{prefix}_cam_bbox.mp4'), fps=2)
    
    # 2) LiDAR on camera video
    make_video_lidar_on_cam(scene_tok, os.path.join(VIDEO_DIR, f'{prefix}_lidar_on_cam.mp4'), fps=2)
    
    # 3) Side-by-side: Camera+BBox | BEV
    make_video_sidebyside(scene_tok, os.path.join(VIDEO_DIR, f'{prefix}_sidebyside.mp4'), fps=2)

# Convert mp4v to H.264 for better compatibility (browsers, etc.)
print(f"\nConverting to H.264 for browser compatibility...")
import subprocess
for f in sorted(os.listdir(VIDEO_DIR)):
    if f.endswith('.mp4'):
        src = os.path.join(VIDEO_DIR, f)
        dst = os.path.join(VIDEO_DIR, f.replace('.mp4', '_h264.mp4'))
        subprocess.run([
            'ffmpeg', '-y', '-i', src, '-c:v', 'libx264', '-preset', 'medium',
            '-crf', '23', '-pix_fmt', 'yuv420p', '-an', dst
        ], capture_output=True)
        # Replace original with H.264 version
        os.replace(dst, src)
        print(f"  Converted: {f}")

print(f"\nAll videos saved to: {VIDEO_DIR}")
for f in sorted(os.listdir(VIDEO_DIR)):
    size_mb = os.path.getsize(os.path.join(VIDEO_DIR, f)) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")

In [ ]:
print(f'\n=== All outputs in: {SAVE_DIR} ===')
print('\n--- Images ---')
for f in sorted(os.listdir(SAVE_DIR)):
    fp = os.path.join(SAVE_DIR, f)
    if os.path.isfile(fp):
        size_mb = os.path.getsize(fp) / 1e6
        print(f'  {f}  ({size_mb:.1f} MB)')

print('\n--- Videos ---')
for f in sorted(os.listdir(VIDEO_DIR)):
    size_mb = os.path.getsize(os.path.join(VIDEO_DIR, f)) / 1e6
    print(f'  {f}  ({size_mb:.1f} MB)')